# 🧾 Doc Agent — Team Collaboration Launcher
### Multi-Modal Document Intelligence Agent

**This notebook is for team collaboration via GitHub.**
It clones the project, installs dependencies, and launches the full Streamlit UI via a public Cloudflare Tunnel URL.

---

## How to use

1. **Run all cells top-to-bottom** (`Runtime → Run all`)
2. Set your `ANTHROPIC_API_KEY` in **Colab Secrets** (🔑 icon in left sidebar → New secret)
3. Cell 5 prints a **public URL** — open it in your browser to see the Streamlit app
4. Edit files in `/content/capstone/` using the Colab file browser
5. Commit and push your changes back with Cell 6

---

> **Login credentials:** `demo` / `demo`

## Step 1 — Clone the Repository

In [ ]:
import os

# ── Set your GitHub repo URL here ────────────────────────────────────────
GITHUB_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git"
PROJECT_DIR = "/content/capstone"
# ─────────────────────────────────────────────────────────────────────────

if os.path.exists(PROJECT_DIR):
    print("Repo already cloned. Pulling latest changes...")
    !git -C {PROJECT_DIR} pull
else:
    print("Cloning repository...")
    !git clone {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!git log --oneline -5

## Step 2 — Install System Dependencies

In [ ]:
%%capture
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils libgl1
print("System dependencies installed.")

## Step 3 — Install Python Packages

In [ ]:
%%capture
!pip install -r /content/capstone/requirements.txt
print("Python packages installed.")

## Step 4 — Configure API Key

In [ ]:
import os, getpass

# Option A: Colab Secrets — add ANTHROPIC_API_KEY via 🔑 sidebar
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option B: type it manually (not stored in notebook)
    api_key = getpass.getpass("Enter your Anthropic API key: ")

os.environ["ANTHROPIC_API_KEY"] = api_key

# Write a .env file so the app picks it up
with open("/content/capstone/.env", "w") as f:
    f.write(f"ANTHROPIC_API_KEY={api_key}\n")
    f.write("VLM_BACKEND=claude\n")
    f.write("CLAUDE_MODEL=claude-sonnet-4-6\n")
    f.write("SQLITE_PATH=/content/capstone/data/invoices.db\n")
    f.write("CHROMA_PERSIST_DIR=/content/capstone/data/chroma\n")

!mkdir -p /content/capstone/data/samples /content/capstone/data/output
print("✅ .env written. Data directories created.")

## Step 5 — Launch Streamlit App

This cell starts the Streamlit server and creates a **public Cloudflare Tunnel URL**.
Share that URL with your team — everyone can open it in their browser.

> **Login:** `demo` / `demo`

> **Note:** The URL is active as long as this Colab session is running. Each new session gets a new URL.

In [ ]:
import subprocess, threading, time, re, sys

# ── 1. Download cloudflared (one-shot binary, no account needed) ──────────
!wget -q -O /usr/local/bin/cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

# ── 2. Start Streamlit in the background ─────────────────────────────────
streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    cwd="/content/capstone",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("⏳ Starting Streamlit server...")
time.sleep(4)

# ── 3. Open Cloudflare Tunnel and capture the public URL ─────────────────
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline().decode("utf-8", errors="ignore")
    match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group()
        break
    time.sleep(1)

if public_url:
    print(f"""
╔══════════════════════════════════════════════════════════╗
║  🚀  Doc Agent is LIVE                                   ║
║                                                          ║
║  URL  : {public_url:<46}║
║  Login: demo / demo                                      ║
║                                                          ║
║  Share this URL with your team.                          ║
║  Active while this Colab session is running.             ║
╚══════════════════════════════════════════════════════════╝
""")
else:
    print("❌ Could not get tunnel URL. Check that port 8501 is running.")
    print("   Alternatively, use the localtunnel cell below.")

### 5b — Alternative: localtunnel (if Cloudflare doesn't work)

In [ ]:
# Only run this if Step 5 didn't produce a URL
import urllib.request
ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()
print(f"When the tunnel asks for a password, enter: {ip}")
!npm install -g localtunnel --silent
!lt --port 8501

---

## Step 6 — Edit Code & Contribute Back

While the app is running, edit any file in `/content/capstone/` using:
- The **Colab file browser** (left sidebar → folder icon)
- Or the cells below

After editing, commit and push your changes:

In [ ]:
# Configure git identity (first time only)
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

In [ ]:
# See what changed
!git -C /content/capstone status
!git -C /content/capstone diff --stat

In [ ]:
COMMIT_MESSAGE = "Your descriptive commit message here"

!git -C /content/capstone add -A
!git -C /content/capstone commit -m "{COMMIT_MESSAGE}"
!git -C /content/capstone push

---

## Step 7 — Restart App After Code Changes

Streamlit auto-reloads when files change — just save the file.
If it doesn't reload, run this cell to force a restart:

In [ ]:
import subprocess, sys, time

# Kill existing Streamlit
!pkill -f streamlit || true
time.sleep(2)

# Restart
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false"],
    cwd="/content/capstone",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
print(f"✅ Streamlit restarted. Refresh {public_url}")

---

## Collaboration Tips

| Task | How |
|------|-----|
| View live app | Open the Cloudflare URL from Step 5 |
| Edit a file | Left sidebar → file browser → click any `.py` file |
| Test a change | Save the file — Streamlit reloads automatically |
| Run pipeline in cells | Use the original `invoice_agent_colab.ipynb` |
| Commit work | Run Step 6 cells |
| Fresh start | `Runtime → Factory reset runtime`, then re-run all |

### Project structure reminder
```
capstone/
├── app.py                ← Streamlit entry point
├── ui/pages/             ← Edit pages here (dashboard, process, review…)
├── agents/               ← Agent logic (extraction, validation, correction)
├── graph/workflow.py     ← LangGraph state machine
└── models/schemas.py     ← Pydantic data models
```